# Projeto - Simulando a Opinião Pública
## Tema: Democracia
## Modelo escolhido: Gemma 3, 1B de parâmetros

## Primeiros passos
### Antes de executar as células de código, siga os passos a seguir
**Baixar e preparar Ollama:**
1. Escolha o comando apropriado para o seu SO (MacOS, Linux, Windows) no site do [Ollama](https://ollama.com/download)
> Para Linux/MacOS, pode ser necessário executar o comando abaixo
> ```bash
>  sudo apt-get install zstd
> ```
2.  Execute o servidor do Ollama localmente, o que deve abrir a porta `localhost:11434`
    ```bash
    ollama serve
    ```
> OBS: Se retornar que a porta já está em uso, é porque após o download, já foi executado automaticamente o comando anterior. Para confirmar, acesse `http://localhost:11434` no navegador e veja se retorna *Ollama is running* 
3. Execute o comando abaixo para baixar e executar o modelo:
    ```bash
    ollama run gemma3:1b "preload" > /dev/null
    ```
> OBS: Se não funcionar, execute `ollama pull gemma3:1b antes`

**Preparar ambiente virtual do Python:**<br><br>
Para baixar as dependências apenas para esse projeto (e não de forma fixa na sua máquina) execute essas instruções:
```bash
python3 -m venv .venv;
source .venv/bin/activate;
```

### Baixando bibliotecas necessárias

* **Pandas:** Para trabalhar com a base de dados
* **PyReadStat:** Para trabalhar com a base de dados, permitindo ler arquivo .sav e complementar o pandas
* **Ollama:** API do LLM
* **Pydantic + Typing**: Validação de formato de saída JSON

In [51]:
# Instalar a lib de Python para Ollama
!pip install ollama

In [52]:
# Instalar pndas
!pip install pandas

In [53]:
# Instalar a lib para ler arquivos .sav
!pip install pyreadstat

In [54]:
!pip install pydantic

### Desenvolvimento da conversa com o modelo

#### Parte 1) Selecionar variáveis mais relevantes
O primeiro passo é pedir ao LLM reconhecer quais são as variáveis mais relevantes para ele na hora de simular o participante.

Para isso, para cada pergunta, será enviado um prompt indicando essa tarefa, as variáveis com valores possíveis, a pergunta e um JSON Schema de resposta esperada.

**Prompt de Sistema**
> "*Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
>
> *Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
> *Características possíveis: <lista de características, ignorando as outras perguntas>*

**Prompt de Entrada**
> ```json
>
>  {
>    pergunta: ""
>    possiveis_respostas: [...],
>    multipla_escolha: true | false
>  }
>```
"

Json de saída esperado:
```json
  {
    "question" : {
      "type": "string"
    },
    "variaveis_selecionadas": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "minItems": 1,
        "uniqueItems": true,
        "default": [
          "Nenhuma é relevante"
        ]
      },
  }
```


In [55]:
import pandas as pd

df = pd.read_spss("04832.SAV")

print("Colunas possíveis:", list(df.columns))

participant_features = list(df.columns[~df.columns.str.contains(r"P\d\D*")])
print("Características dos participantes:", participant_features)

Colunas possíveis: ['SEXO', 'IDADE', 'FX_ID', 'ESCOLARIDADE', 'P1A', 'P1B', 'P1C', 'P2_1', 'P2_2', 'P2_3', 'P3_1', 'P3_2', 'P3_3', 'P3_4', 'P3_5', 'P3_6', 'P4', 'RACA', 'RELIGIAO', 'REND1', 'REND2', 'REGIAO', 'COND', 'PORTE', 'ID_Ipec', 'DATA_ENTREVISTA', 'TIPO_COLETA']
Características dos participantes: ['SEXO', 'IDADE', 'FX_ID', 'ESCOLARIDADE', 'RACA', 'RELIGIAO', 'REND1', 'REND2', 'REGIAO', 'COND', 'PORTE', 'ID_Ipec', 'DATA_ENTREVISTA', 'TIPO_COLETA']


In [56]:
questions = [
        {
            "pergunta": "Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?",
            "possiveis_respostas": ["Sim", "Não", "Não Votou"],
            "multipla_escolha": False
        },
        {
            "pergunta": "Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?",
            "possiveis_respostas": ["Sim", "Não", "Não Votou"],
            "multipla_escolha": False
        },
        {
            "pergunta": "Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?",
            "possiveis_respostas": ["Sim", "Não", "Não Votou"],
            "multipla_escolha": False
        },
        {
            "pergunta": " Algumas pessoas dizem que a divulgação de fake news, ou seja, de notícias ou conteúdos falsos podem prejudicar a democracia. "
                        "Pensando nisso, quais dessas opções você acredita que poderiam contribuir no combate à divulgação de fake news? ",
            "possiveis_respostas": [
                "Ampliar a regulamentação, as regras a serem cumpridas pelas plataformas digitais (empresas de tecnologia como Facebook, Youtube, WhatsApp, Twitter/X, etc.)",
                "Responsabilizar e punir empresas de tecnologia e de comunicação que não removerem postagens com notícias ou conteúdos falsos",
                "Ampliar a regulamentação, as regras a serem cumpridas pelos usuários que divulgam ou compartilham fake news, criadas por eles próprios ou por terceiros",
                "Responsabilizar e punir os usuários que divulgam ou compartilham postagens com notícias ou conteúdos falsos",
                "Ampliar a regulamentação, as regras a serem cumpridas por políticos/candidatos que divulgam ou compartilham fake news, criadas por eles próprios ou por terceiros",
                "Responsabilizar, punir ou cassar políticos/candidatos que divulgam ou compartilham postagens com notícias ou conteúdos falsos",
                "Não sabe",
                "Não respondeu"
            ],
            "multipla_escolha": True
        }
]

In [57]:
from typing import List
from pydantic import BaseModel, Field

class ExpectedAnswerVariables(BaseModel):
    variaveis_selecionadas: List[str] = Field(
        default_factory=lambda: ["Nenhuma é relevante"],
        min_items=1
    )

/tmp/ipykernel_2419/3041399866.py:5: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  variaveis_selecionadas: List[str] = Field(


In [58]:
import ollama


#Definindo Host do Ollama
ollama_host = 'http://localhost:11434'
ollama.Client(host=ollama_host)

# Usando prompt de sistema
system_prompt = f"""
Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
Características possíveis: {participant_features}
IMPORTANTE: Você deve apenas retornar a lista de características selecionadas, sem explicações ou justificativas. NÃO adicione nenhuma caracterísitca que não esteja na lista.
"""

# Chama da API do Ollama
for question in questions:
    user_question = f"""
    Pergunta: {question['pergunta']}
    Possíveis respostas: {', '.join(question['possiveis_respostas'])}
    Múltipla escolha: {'Sim' if question['multipla_escolha'] else 'Não'}
    """
    try:
        response = ollama.chat(
            model='gemma3:1b',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_question},
            ],
            format=ExpectedAnswerVariables.model_json_schema(),
            options={"temperature": 0.3}
        )
        print("#" * 50)
        print(f"Pergunta Lida: {question['pergunta']}")
        print("Resposta do Ollama:\n")
        print(response['message']['content'])
        print("#" * 50)
    except Exception as e:
        print(f"Ocorreu um erro ao conectar com o Ollama: {e}")
        print("Por favor, certifique-se de que o servidor Ollama está em execução e o modelo 'gemm-3-1B' está baixado.")
        print(f"Você pode tentar iniciar o servidor Ollama e baixar o modelo com: 'ollama run gemm-3-1B' em seu terminal.")


##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?
Resposta do Ollama:

{
  "variaveis_selecionadas": ["SEXO", "IDADE", "FX_ID", "ESCOLARIDADE", "RACA", "RELIGIAO", "REND1", "REND2", "REGIAO", "COND", "PORTE", "ID_Ipec", "DATA_ENTREVISTA", "TIPO_COLETA"]
}
##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

{
  "variaveis_selecionadas": [
    "ESCOLARIDADE"
  ]
}
##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

{
  "variaveis_selecionadas": [
    "ESCOLARIDADE"
  ]
}
##################################################
######################################